# Phase 4: the exact $E_8$ benchmark

We construct the $E_8$ lattice in real dimension eight, certify an integral order-four complex structure and principal Riemann form, and compute the type $(2,2,2,2)$ GKP relative systole by two independent algorithms.

In [1]:
from pathlib import Path
from time import perf_counter
import sys

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))

from gkp_systole import (
    D4_PERIOD_MODEL, E8_COMPLEX_STRUCTURE, E8_GRAM, E8_PPAV_MODEL,
    E8_PRINCIPAL_ALTERNATING, Polarization, repeated_period_model,
    validate_e8_derivation,
)

## Exact PPAV certificate

The Euclidean $E_8$ Gram matrix has determinant one. The paired quarter-turn preserves the lattice and is integral in our basis. The validator independently derives $A=GJ$ and checks that it is a principal polarization.

In [2]:
validate_e8_derivation()
certificate = E8_PPAV_MODEL.validation_certificate()
for key, value in certificate.as_dict().items():
    print(f'{key:32s}: {value}')

dimension_g                     : 4
polarization_type               : (1, 1, 1, 1)
principal                       : True
scale_radicand                  : 1
metric_core_determinant         : 1
physical_metric_determinant     : 1
polarization_determinant        : 1
certified                       : True
checks                          : ('positive-definite exact metric', 'J_num^2 = -r I', 'J_num^T G_core J_num = r G_core', 'A = G_core J_num / r is integral alternating', 'polarization type verified', 'det(G_core)/r^g = |det(A)|')


In [3]:
print('G =')
for row in E8_GRAM:
    print(row)
print('\nJ =')
for row in E8_COMPLEX_STRUCTURE:
    print(row)
print('\nA = GJ =')
for row in E8_PRINCIPAL_ALTERNATING:
    print(row)

G =
(2, 0, 0, 0, 0, 0, 0, 1)
(0, 2, -1, 0, 0, 0, 0, 0)
(0, -1, 2, -1, 0, 0, 0, 0)
(0, 0, -1, 2, -1, 0, 0, 0)
(0, 0, 0, -1, 2, -1, 0, 0)
(0, 0, 0, 0, -1, 2, -1, -1)
(0, 0, 0, 0, 0, -1, 2, 0)
(1, 0, 0, 0, 0, -1, 0, 2)

J =
(1, 2, 0, 0, 0, 0, 0, 0)
(-1, -1, 0, 0, 0, 0, 0, 0)
(-1, -2, -1, 1, 0, 0, 0, 0)
(-2, -2, -2, 1, 0, 0, 0, 0)
(-2, -3, -2, 1, -1, 1, 0, 0)
(-3, -4, -2, 2, -2, 1, 0, 0)
(-1, -2, -1, 1, -1, 0, 0, 1)
(-2, -3, -1, 1, -1, 1, -1, 0)

A = GJ =
(0, 1, -1, 1, -1, 1, -1, 0)
(-1, 0, 1, -1, 0, 0, 0, 0)
(1, -1, 0, 1, 0, 0, 0, 0)
(-1, 1, -1, 0, 1, -1, 0, 0)
(1, 0, 0, -1, 0, 1, 0, 0)
(-1, 0, 0, 1, -1, 0, 1, -1)
(1, 0, 0, 0, 0, -1, 0, 2)
(0, 0, 0, 0, 0, 1, -2, 0)


## Type $(2,2,2,2)$ result

The principal form is doubled for the qubit code. The uniform identity gives $\ell^2=\lambda_1^2/4$.

In [4]:
qubit_polarization = Polarization(E8_PPAV_MODEL.uniform_alternating(2))
shortcut_started = perf_counter()
shortcut = E8_PPAV_MODEL.compute_qubit_systole()
shortcut_seconds = perf_counter() - shortcut_started

print('polarization type :', qubit_polarization.type)
print('kernel order      :', qubit_polarization.kernel_order)
print('lambda_1^2       :', shortcut.lambda1_squared)
print('ell^2            :', shortcut.squared_systole)
print('ell              :', shortcut.systole)
print('N_class          :', shortcut.class_multiplicity)
print('N_lift           :', shortcut.lift_multiplicity)
print('SVP certified    :', shortcut.certified)
print(f'SVP time         : {shortcut_seconds:.4f} s')

polarization type : (2, 2, 2, 2)
kernel order      : 256
lambda_1^2       : 2
ell^2            : 1/2
ell              : 0.7071067811865476
N_class          : 120
N_lift           : 240
SVP certified    : True
SVP time         : 0.0147 s


## Independent full-kernel check

The general pathway enumerates all $2^8-1=255$ nonzero kernel classes and solves a closest-vector problem for each. It must reproduce the SVP result, including both multiplicities.

In [5]:
full_started = perf_counter()
full = E8_PPAV_MODEL.compute_full_uniform_systole(2)
full_seconds = perf_counter() - full_started
assert full.squared_systole == shortcut.squared_systole
assert full.class_multiplicity == shortcut.class_multiplicity
assert full.lift_multiplicity == shortcut.lift_multiplicity
print('full ell^2       :', full.squared_systole)
print('full N_class     :', full.class_multiplicity)
print('full N_lift      :', full.lift_multiplicity)
print('full certified   :', full.certified)
print(f'full CVP time    : {full_seconds:.4f} s')
print('Independent paths agree exactly.')

full ell^2       : 1/2
full N_class     : 120
full N_lift      : 240
full certified   : True
full CVP time    : 0.6863 s
Independent paths agree exactly.


## Comparison with the decomposable $D_4^2$ baseline

In [6]:
d4_squared = repeated_period_model(D4_PERIOD_MODEL, 2).compute_uniform_systole(2)
print(f"D4^2 ell^2 : {d4_squared.squared_systole:.12f}")
print(f"E8   ell^2 : {float(shortcut.squared_systole):.12f}")
print(f"ratio       : {float(shortcut.squared_systole) / d4_squared.squared_systole:.6f}")
assert float(shortcut.squared_systole) > d4_squared.squared_systole

D4^2 ell^2 : 0.353553390593
E8   ell^2 : 0.500000000000
ratio       : 1.414214


## Conclusion

The exact result is

$$\boxed{\lambda_1^2=2,\quad \ell^2=1/2,\quad N_{\rm class}=120,\quad N_{\rm lift}=240.}$$

Viazovska proved that $E_8$ is the densest packing in eight dimensions ([arXiv:1603.04246](https://arxiv.org/abs/1603.04246)). Since our construction certifies that the covolume-one $E_8$ metric is a PPAV metric, this attains the global bound $\ell^2\le 1/2$ for the fixed-principal type $(2,2,2,2)$ problem. It is also another CM optimizer: the order-four complex structure gives Gaussian CM.